# 조합C + Daily_Return 모델 4종 비교

Daily_Return 추가 조건에서 실행한 네 모델의 전체 OOS 지표를 비교한다. 각 숫자는 같은 HF
parquet 리비전과 같은 12개 expanding 검증 구간에서 나온 실행 결과다.


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")

combination = "C"
variant = "daily"
return_features = ('daily_return',)
model_names = ('LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM')
payloads = []
for model_name in model_names:
    path = project_root / "data" / "raw" / "model_results" / (
        f"{combination}_{variant}_{model_name}.json"
    )
    if not path.is_file():
        raise FileNotFoundError(f"먼저 01~04 모델 노트북을 실행해야 합니다: {path}")
    payloads.append(json.loads(path.read_text(encoding="utf-8")))

# 서로 다른 데이터나 피처 실험의 숫자가 한 표에 섞이면 즉시 중단합니다.
source_keys = {
    (item["source"]["repo_sha"], item["source"]["index_sha256"])
    for item in payloads
}
experiment_keys = {
    (
        item["experiment"]["combination"],
        tuple(item["experiment"]["return_features"]),
    )
    for item in payloads
}
if len(source_keys) != 1 or experiment_keys != {(combination, return_features)}:
    raise RuntimeError("네 모델의 HF 출처 또는 피처 구성이 서로 다릅니다.")

rows = []
for item in payloads:
    summary = item["summary"]
    rows.append(
        {
            "모델": summary["model"],
            "Accuracy": summary["accuracy"],
            "Macro F1": summary["macro_f1"],
            "하락 Recall": summary["down_recall"],
            "핵심지표 조화평균": summary["core_harmonic_mean"],
            "ΔSharpe_net 중앙값": summary["delta_sharpe_net_median"],
            "현금 폴드": summary["all_cash_folds"],
        }
    )
comparison_df = pd.DataFrame(rows).sort_values(
    "핵심지표 조화평균", ascending=False, kind="stable"
).reset_index(drop=True)
display(comparison_df.round(4))

best = comparison_df.iloc[0]
display(Markdown(f"""
## 자동 비교 결과

- 세 핵심지표 조화평균 1위: **{best['모델']}** `{best['핵심지표 조화평균']:.4f}`
- Accuracy: `{best['Accuracy']:.4f}`
- Macro F1: `{best['Macro F1']:.4f}`
- 하락 Recall: `{best['하락 Recall']:.4f}`
- 비용 차감 ΔSharpe 폴드 중앙값: `{best['ΔSharpe_net 중앙값']:.4f}`

순위는 합의한 기준인 Accuracy·Macro F1·하락 Recall 조화평균으로 정했다.
"""))


,모델,Accuracy,Macro F1,하락 Recall,핵심지표 조화평균,ΔSharpe_net 중앙값,현금 폴드
0,XGBoost,0.3875,0.3676,0.2887,0.3423,-0.1777,0
1,LightGBM,0.3764,0.3626,0.2784,0.3331,0.0043,0
2,RandomForest,0.3708,0.3586,0.2784,0.3305,0.1140,0
3,LogisticRegression,0.4014,0.3759,0.2371,0.3202,0.1897,1



## 자동 비교 결과

- 세 핵심지표 조화평균 1위: **XGBoost** `0.3423`
- Accuracy: `0.3875`
- Macro F1: `0.3676`
- 하락 Recall: `0.2887`
- 비용 차감 ΔSharpe 폴드 중앙값: `-0.1777`

순위는 합의한 기준인 Accuracy·Macro F1·하락 Recall 조화평균으로 정했다.


## 개별 실험

- [Logistic Regression](01.LogisticRegression.ipynb)
- [RandomForest](02.RandomForest.ipynb)
- [XGBoost](03.XGBoost.ipynb)
- [LightGBM](04.LightGBM.ipynb)